In [ ]:
!pip install spikingjelly -q
!pip install decord -q
!pip install brian2 -q
import os
import math
from sklearn.manifold import TSNE
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np
import torchvision.transforms as transforms
from PIL import Image
from spikingjelly.activation_based import base, layer, neuron, surrogate, functional
import pandas as pd
from pathlib import Path
import random
from torch.utils.data import Dataset, DataLoader
from decord import VideoReader, cpu
import decord
from random import choice
from tqdm import tqdm
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, f1_score, precision_score,recall_score 
import itertools
from brian2 import *
import seaborn as sns
import json
import cv2
import torch.nn.functional as F
device = torch.device(
        'cuda' if torch.cuda.is_available() else 'cpu'
    )

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 437.6/437.6 kB 7.6 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 85.1 MB/s eta 0:00:00:00:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is inco

In [ ]:
class ConvRecurrentContainer(base.MemoryModule):
    def __init__(self, sub_module, in_channels, out_channels, stride, step_mode="s"):
        super().__init__()
        self.step_mode = step_mode
        assert not hasattr(sub_module, "step_mode") or sub_module.step_mode == "s"
        self.sub_module_out_channels = out_channels
        self.sub_module = sub_module

        if stride > 1:
            self.dwconv = nn.ConvTranspose2d(
                out_channels, out_channels,
                kernel_size=3, stride=stride,
                padding=1, output_padding=1,
                groups=out_channels, bias=False
            )
            self.bn1 = layer.BatchNorm2d(out_channels)
            self.sn1 = neuron.LIFNode(
                surrogate_function=surrogate.ATan(),
                detach_reset=True
            )
        else:
            self.dwconv = None

        self.pwconv = nn.Conv2d(
            in_channels + out_channels,
            in_channels,
            kernel_size=1, stride=1, bias=False
        )
        self.bn = layer.BatchNorm2d(in_channels)
        self.sn = neuron.LIFNode(
            surrogate_function=surrogate.ATan(),
            detach_reset=True
        )

        self.register_memory("y", None)

    def single_step_forward(self, x):
        if self.y is None:
            if self.dwconv is None:
                self.y = torch.zeros(
                    x.size(0), self.sub_module_out_channels, x.size(2), x.size(3),
                    device=x.device, dtype=x.dtype
                )
            else:
                h = (x.size(2) + 2 - (3 - 1) - 1) // 2 + 1
                w = (x.size(3) + 2 - (3 - 1) - 1) // 2 + 1
                self.y = torch.zeros(
                    x.size(0), self.sub_module_out_channels, h, w,
                    device=x.device, dtype=x.dtype
                )

        if self.dwconv is None:
            out = self.y
        else:
            out = self.sn1(self.bn1(self.dwconv(self.y)))

        out = torch.cat((x, out), dim=1)
        out = self.bn(self.pwconv(out))
        out = self.sn(out)
        self.y = self.sub_module(out)

        return self.y


def sew_function(x, y, cnf):
    if cnf == "ADD":
        return x + y
    elif cnf == "AND":
        return x * y
    elif cnf == "OR":
        return x + y - x * y
    else:
        raise NotImplementedError


def conv3x3(in_planes, out_planes, stride=1, groups=1, dilation=1):
    return layer.Conv2d(
        in_planes, out_planes, kernel_size=3, stride=stride,
        padding=dilation, groups=groups, bias=False, dilation=dilation
    )


def conv1x1(in_planes, out_planes, stride=1):
    return layer.Conv2d(in_planes, out_planes, kernel_size=1, stride=stride, bias=False)


class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_planes, planes, stride=1, downsample=None, groups=1,
                 base_width=64, norm_layer=None, cnf=None):
        super().__init__()
        if norm_layer is None:
            norm_layer = layer.BatchNorm2d

        if groups != 1 or base_width != 64:
            raise ValueError("BasicBlock only supports groups=1 and base_width=64")

        self.conv1 = conv3x3(in_planes, planes, stride)
        self.bn1 = norm_layer(planes)
        self.sn1 = neuron.LIFNode(
            surrogate_function=surrogate.ATan(),
            detach_reset=True
        )

        self.conv2 = conv3x3(planes, planes)
        self.bn2 = norm_layer(planes)
        self.sn2 = neuron.LIFNode(
            surrogate_function=surrogate.ATan(),
            detach_reset=True
        )

        self.downsample = downsample
        if downsample is not None:
            self.downsample_sn = neuron.LIFNode(
                surrogate_function=surrogate.ATan(),
                detach_reset=True
            )

        self.stride = stride
        self.cnf = cnf

    def forward(self, x):
        identity = x

        out = self.sn1(self.bn1(self.conv1(x)))
        out = self.sn2(self.bn2(self.conv2(out)))

        if self.downsample is not None:
            identity = self.downsample_sn(self.downsample(x))

        out = sew_function(out, identity, self.cnf)
        return out


class Bottleneck(nn.Module):
    expansion = 4

    def __init__(self, in_planes, planes, stride=1, downsample=None, groups=1,
                 base_width=64, norm_layer=None, cnf=None):
        super().__init__()
        if norm_layer is None:
            norm_layer = layer.BatchNorm2d

        width = int(planes * (base_width / 64.0)) * groups

        self.conv1 = conv1x1(in_planes, width)
        self.bn1 = norm_layer(width)
        self.sn1 = neuron.LIFNode(
            surrogate_function=surrogate.ATan(),
            detach_reset=True
        )

        self.conv2 = conv3x3(width, width, stride, groups)
        self.bn2 = norm_layer(width)
        self.sn2 = neuron.LIFNode(
            surrogate_function=surrogate.ATan(),
            detach_reset=True
        )

        self.conv3 = conv1x1(width, planes * self.expansion)
        self.bn3 = norm_layer(planes * self.expansion)
        self.sn3 = neuron.LIFNode(
            surrogate_function=surrogate.ATan(),
            detach_reset=True
        )

        self.downsample = downsample
        if downsample is not None:
            self.downsample_sn = neuron.LIFNode(
                surrogate_function=surrogate.ATan(),
                detach_reset=True
            )

        self.stride = stride
        self.cnf = cnf

    def forward(self, x):
        identity = x

        out = self.sn1(self.bn1(self.conv1(x)))
        out = self.sn2(self.bn2(self.conv2(out)))
        out = self.sn3(self.bn3(self.conv3(out)))

        if self.downsample is not None:
            identity = self.downsample_sn(self.downsample(x))

        out = sew_function(out, identity, self.cnf)
        return out


class LoRaFBSNet(nn.Module):
    def __init__(self, block, layers, num_classes=10, groups=1, width_per_groups=64,
                 norm_layer=None, cnf="ADD", zero_init_residual=False):
        super().__init__()

        if norm_layer is None:
            norm_layer = layer.BatchNorm2d
        self._norm_layer = norm_layer

        self.in_planes = 64
        self.groups = groups
        self.base_width = width_per_groups

        self.conv1 = layer.Conv2d(1, self.in_planes, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = self._norm_layer(self.in_planes)
        self.sn1 = neuron.LIFNode(surrogate_function=surrogate.ATan(), detach_reset=True)
        self.maxpool = layer.MaxPool2d(kernel_size=3, stride=2, padding=1)

        layer1 = self._make_layer(block, 64, layers[0], cnf=cnf)
        self.recurrent_layer1 = ConvRecurrentContainer(
            layer1, in_channels=64, out_channels=64 * block.expansion, stride=1
        )

        layer2 = self._make_layer(block, 128, layers[1], stride=2, cnf=cnf)
        self.recurrent_layer2 = ConvRecurrentContainer(
            layer2, in_channels=64 * block.expansion,
            out_channels=128 * block.expansion, stride=2
        )

        layer3 = self._make_layer(block, 256, layers[2], stride=2, cnf=cnf)
        self.recurrent_layer3 = ConvRecurrentContainer(
            layer3, in_channels=128 * block.expansion,
            out_channels=256 * block.expansion, stride=2
        )

        layer4 = self._make_layer(block, 512, layers[3], stride=2, cnf=cnf)
        self.recurrent_layer4 = ConvRecurrentContainer(
            layer4, in_channels=256 * block.expansion,
            out_channels=512 * block.expansion, stride=2
        )

        self.avgpool = layer.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512 * block.expansion, num_classes)


        for m in self.modules():
            if isinstance(m, (layer.Conv2d, nn.Conv2d, nn.ConvTranspose2d)):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, layer.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

        if zero_init_residual:
            for m in self.modules():
                if isinstance(m, Bottleneck):
                    nn.init.constant_(m.bn3.weight, 0)
                elif isinstance(m, BasicBlock):
                    nn.init.constant_(m.bn2.weight, 0)

    def _make_layer(self, block, planes, num_blocks, stride=1, cnf=None):
        downsample = None
        if stride != 1 or self.in_planes != planes * block.expansion:
            downsample = nn.Sequential(
                conv1x1(self.in_planes, planes * block.expansion, stride),
                self._norm_layer(planes * block.expansion)
            )

        layers_list = [
            block(
                self.in_planes, planes, stride, downsample,
                self.groups, self.base_width, self._norm_layer, cnf
            )
        ]

        self.in_planes = planes * block.expansion

        for _ in range(1, num_blocks):
            layers_list.append(
                block(
                    self.in_planes, planes,
                    groups=self.groups,
                    base_width=self.base_width,
                    norm_layer=self._norm_layer,
                    cnf=cnf
                )
            )

        return nn.Sequential(*layers_list)

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.sn1(x)
        x = self.maxpool(x)

        x = self.recurrent_layer1(x)
        x = self.recurrent_layer2(x)
        x = self.recurrent_layer3(x)
        x = self.recurrent_layer4(x)

        x = self.avgpool(x)

        if self.avgpool.step_mode == 's':
            x = torch.flatten(x, 1)
        elif self.avgpool.step_mode == 'm':
            x = torch.flatten(x, 2)

        x = self.fc(x)
        return x



def lorafb_snet18(**kwargs):
    return LoRaFBSNet(BasicBlock, [2, 2, 2, 2], **kwargs)

In [ ]:
# ─────────────────────────────────────────────────────────────
class LoRaFBSNetFeatureExtractor(nn.Module):
    def __init__(self, full_model):
        super().__init__()
        self.conv1 = full_model.conv1
        self.bn1 = full_model.bn1
        self.sn1 = full_model.sn1
        self.maxpool = full_model.maxpool
        self.layer1 = full_model.recurrent_layer1
        self.layer2 = full_model.recurrent_layer2
        self.layer3 = full_model.recurrent_layer3
        self.layer4 = full_model.recurrent_layer4
        self.avgpool = full_model.avgpool

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.sn1(x)
        x = self.maxpool(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)
        return torch.flatten(x, 2)   # (T, B, D)


def set_step_mode(net, step_mode, keep_instance):
    keep_step_mode_instance = (
        layer.StepModeContainer, layer.ElementWiseRecurrentContainer, layer.LinearRecurrentContainer
    )
    keep_step_mode_instance += keep_instance
    # step_mode of sub-modules in keep_step_mode_instance will not be changed

    keep_step_mode_containers = []
    for m in net.modules():
        if isinstance(m, keep_step_mode_instance):
            keep_step_mode_containers.append(m)

    for m in net.modules():
        if hasattr(m, "step_mode"):
            is_contained = False
            for container in keep_step_mode_containers:
                if not isinstance(m, keep_step_mode_instance) and m in container.modules():
                    is_contained = True
                    break
            if is_contained:
                # this function should not change step_mode of submodules in keep_step_mode_containers
                pass
            else:
                m.step_mode = step_mode

def topk_selective_neurons(
    X_TCx,
    y,
    K=16,
    use_dprime=True,
    margin_thr=0.001,
    act_thr=1e-4,
    auto_relax=True,
    relax_factor=0.5,
    max_relax_steps=5,
    min_margin=0.0,
    min_act=0.0
):
    """
    Selects top-K selective neurons per class using per-class preferred matching, 
    d' (Signal-to-Noise Ratio) or Margin scoring, and per-class Auto-Relaxation.
    
    Inputs
    ------
    X_TCx : (N, T, C) array of activities/spikes
    y     : (N,) class labels
    K     : number of top neurons to select per class (equivalent to topk)
    """
    classes = np.unique(y)
    num_classes = len(classes)
    N, T, C = X_TCx.shape

    # 1. Temporal Averaging (N, C)
    feat = X_TCx.mean(axis=1)

    mu = np.zeros((num_classes, C), dtype=np.float32)
    var = np.zeros((num_classes, C), dtype=np.float32)

    # 2. Per-class mean & variance
    for idx, c in enumerate(classes):
        mask = (y == c)
        if not np.any(mask):
            mu[idx] = 0.0
            var[idx] = 1.0
            continue
        mu[idx] = feat[mask].mean(axis=0)
        var[idx] = feat[mask].var(axis=0) + 1e-6

    # 3. Class preference & local d' / Margin scoring
    best_class = mu.argmax(axis=0)       # (C,) preferred class per neuron
    mu_sorted = np.sort(mu, axis=0)     # (num_classes, C)
    mu_best = mu_sorted[-1, :]
    mu_second = mu_sorted[-2, :]
    margin = mu_best - mu_second     # (C,)

    if use_dprime:
        var_best = np.take_along_axis(var, best_class[None, :], axis=0)[0]
        var_not_best = (var.sum(axis=0) - var_best) / max(1, num_classes - 1)
        dprime_local = (mu_best - mu_second) / \
            np.sqrt(0.5 * (var_best + var_not_best))
        score_all = dprime_local
    else:
        score_all = margin

    per_class_indices = []
    per_class_scores = {}

    # 4. Selection loop with adaptive threshold relaxation
    for c_idx, c in enumerate(classes):
        cand_all = np.where(best_class == c_idx)[0]

        # In case no neuron naturally prefers this class
        if cand_all.size == 0:
            per_class_indices.append(np.array([], dtype=int))
            per_class_scores[f'score{c_idx}'] = score_all
            continue

        cand_margin_full = margin[cand_all]
        cand_mu_full = mu[c_idx, cand_all]

        m_thr = margin_thr
        a_thr = act_thr
        selected_for_class = np.array([], dtype=int)

        for step in range(max_relax_steps + 1):
            keep = (cand_margin_full >= m_thr) & (cand_mu_full >= a_thr)
            cand = cand_all[keep]

            if cand.size > 0:
                cand_scores = score_all[cand]
                order = np.argsort(-cand_scores)
                selected_for_class = cand[order][:K]
                break

            if not auto_relax or step == max_relax_steps:
                break

            # Relax thresholds dynamically
            m_thr = max(min_margin, m_thr * relax_factor)
            a_thr = max(min_act, a_thr * relax_factor)

        per_class_indices.append(selected_for_class.astype(int))
        per_class_scores[f'score{c_idx}'] = score_all

    # 5. Format output exact to target dictionary structure
    res = {
        'per_class': per_class_indices,
        'class_means': mu,
        'fisher_score': score_all,  # Holds d' score (or margin score)
    }
    res.update(per_class_scores)
    return res


def save_neuron_selection(sel, path='neuron_selection.npz'):

    save_dict = {
        'class_means': sel['class_means'],
    }

    for i, v in enumerate(sel['per_class']):
        save_dict[f'class_{i}'] = v

    np.savez(path, **save_dict)
    print(f"Neuron selection saved → {path}")


def load_neuron_selection(path='neuron_selection.npz'):

    d = np.load(path)
    n = sum(1 for k in d if k.startswith('class_'))
    per_class = [d[f'class_{i}'] for i in range(n)]

    global_topk = np.unique(np.concatenate(per_class))

    return {
        'global': global_topk,
        'class_means': d['class_means'],
        'per_class': per_class
    }


# ─────────────────────────────────────────────────────────────
#  VISUALIZATIONS (COMPATIBLE WITH (N, T, C))
# ─────────────────────────────────────────────────────────────

def load_model_from_state_dict(save_path, num_classes):
    model = lorafb_snet18(num_classes=num_classes, cnf="ADD")
    checkpoint = torch.load(save_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    return model


0.0 GB Allocated
0.0 GB Reserved
Epoch   1 | loss=0.6192 | train_acc=0.6396 | val_acc=0.6346 | gap=0.0050
Epoch   2 | loss=0.5762 | train_acc=0.6883 | val_acc=0.7308 | gap=0.0425
Epoch   3 | loss=0.5548 | train_acc=0.7208 | val_acc=0.6923 | gap=0.0285
Epoch   4 | loss=0.5596 | train_acc=0.7110 | val_acc=0.7500 | gap=0.0390
Epoch   5 | loss=0.5067 | train_acc=0.6948 | val_acc=0.7885 | gap=0.0937
Epoch   6 | loss=0.4587 | train_acc=0.7825 | val_acc=0.7500 | gap=0.0325
Epoch   7 | loss=0.4922 | train_acc=0.7532 | val_acc=0.8462 | gap=0.0929
Epoch   8 | loss=0.4638 | train_acc=0.7890 | val_acc=0.9231 | gap=0.1341
Epoch   9 | loss=0.3416 | train_acc=0.8701 | val_acc=0.8846 | gap=0.0145
Epoch  10 | loss=0.2765 | train_acc=0.8994 | val_acc=0.9038 | gap=0.0045
Epoch  11 | loss=0.3183 | train_acc=0.8669 | val_acc=0.9038 | gap=0.0370
Epoch  12 | loss=0.1995 | train_acc=0.9156 | val_acc=0.8846 | gap=0.0310


KeyboardInterrupt: 

In [ ]:
class CustomTripletDataset(Dataset):
    def __init__(self, json_path, video_dir, num_frames=16, transform=None):

        with open(json_path, 'r', encoding='utf-8') as f:
            self.triplets = json.load(f)

        self.video_dir = video_dir
        self.num_frames = num_frames
        self.transform = transform or transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.Grayscale(num_output_channels=1),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485], std=[0.224])
        ])

    def _load_frames(self, video_name):
        video_path = os.path.join(self.video_dir, video_name)
        if not os.path.exists(video_path):
            raise FileNotFoundError(f"Video Not Found:{video_path}")

        cap = cv2.VideoCapture(video_path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

        if total_frames <= 0:
            raise ValueError(f"Videos Frames are not Readable:  {video_path}")

        indices = np.linspace(0, total_frames - 1, self.num_frames, dtype=int)
        frames = []

        for idx in indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
            ret, frame = cap.read()
            if not ret:
                break
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            pil_img = Image.fromarray(frame)
            if self.transform:
                pil_img = self.transform(pil_img)
            frames.append(pil_img)

        cap.release()

        video_tensor = torch.stack(frames, dim=1)
        return video_tensor

    def __len__(self):
        return len(self.triplets)

    def __getitem__(self, idx):
        item = self.triplets[idx]
        triplet_id = item['triplet_id']
        video_list = item['videos']  # [Anchor, Positive, Negative]

        v_a = self._load_frames(video_list[0])
        v_p = self._load_frames(video_list[1])
        v_n = self._load_frames(video_list[2])

        return {
            'triplet_id': triplet_id,
            'anchor': v_a,
            'positive': v_p,
            'negative': v_n,
            'video_names': video_list
        }


dataset = CustomTripletDataset(
    json_path="/kaggle/input/datasets/mohammadmehranfar/triplet-dataset/Triplet_data/Triplet_train.json",
    video_dir="/kaggle/input/datasets/mohammadmehranfar/isik-dataset/dyad_videos_3000ms",
    num_frames=16
)

In [ ]:
# def train_lora_feature_extractor(model, train_loader, optimizer, criterion, epochs=2, save_dir="./lora_checkpoints"):
#     model.to(device)
#     model.train()
#     os.makedirs(save_dir, exist_ok=True)
#     best_loss = float('inf')

#     print("\n🔥 Starting LoRA Training Phase...")
#     print(f"📁 Checkpoints will be saved in: {save_dir}\n")

#     for epoch in range(epochs):
#         running_loss = 0.0
#         pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")

#         for batch in pbar:
#             v_a = batch['anchor'].to(device)
#             v_p = batch['positive'].to(device)
#             v_n = batch['negative'].to(device)

#             v_a = v_a.permute(2, 0, 1, 3, 4).contiguous()
#             v_p = v_p.permute(2, 0, 1, 3, 4).contiguous()
#             v_n = v_n.permute(2, 0, 1, 3, 4).contiguous()

#             optimizer.zero_grad()

#             # Pass Anchor
#             functional.reset_net(model)
#             out_a = model(v_a)
#             z_a = out_a.mean(dim=0) if out_a.dim() > 2 else out_a
#             if z_a.dim() == 1: z_a = z_a.unsqueeze(0)
#             z_a = F.normalize(z_a, p=2, dim=-1)

#             # Pass Positive
#             functional.reset_net(model)
#             out_p = model(v_p)
#             z_p = out_p.mean(dim=0) if out_p.dim() > 2 else out_p
#             if z_p.dim() == 1: z_p = z_p.unsqueeze(0)
#             z_p = F.normalize(z_p, p=2, dim=-1)

#             # Pass Negative
#             functional.reset_net(model)
#             out_n = model(v_n)
#             z_n = out_n.mean(dim=0) if out_n.dim() > 2 else out_n
#             if z_n.dim() == 1: z_n = z_n.unsqueeze(0)
#             z_n = F.normalize(z_n, p=2, dim=-1)

#             loss = criterion(z_a, z_p, z_n)
#             loss.backward()
#             optimizer.step()

#             running_loss += loss.item()
#             pbar.set_postfix({'loss': f"{loss.item():.4f}"})

#             del v_a, v_p, v_n, out_a, out_p, out_n, z_a, z_p, z_n, loss
#             torch.cuda.empty_cache()

#         epoch_loss = running_loss / len(train_loader)
#         print(f"\n📉 Epoch [{epoch+1}/{epochs}] - Triplet Loss: {epoch_loss:.4f}")

#         checkpoint_path = os.path.join(save_dir, f"lora_checkpoint_epoch_{epoch+1}.pth")
#         torch.save({
#             'epoch': epoch + 1,
#             'model_state_dict': model.state_dict(),
#             'optimizer_state_dict': optimizer.state_dict(),
#             'loss': epoch_loss,
#         }, checkpoint_path)
#         print(f"💾 Saved: {checkpoint_path}")

#         if epoch_loss < best_loss:
#             best_loss = epoch_loss
#             best_model_path = os.path.join(save_dir, "lora_best_model.pth")
#             torch.save(model.state_dict(), best_model_path)
#             print(f"🌟 New Best Model Saved (Loss: {best_loss:.4f}) -> {best_model_path}")

#         print("-" * 50)

#     print("✅ LoRA Training Completed Successfully!")
#     return model

@torch.no_grad()
def extract_and_save_wang_inputs(model, dataloader, save_json_path):
    results = []
    print(f"\n🚀 Extracting Features & Firing Rates -> {save_json_path}")

    for batch in tqdm(dataloader, desc="Extracting"):
        v_a = batch['anchor'].to(device).permute(2, 0, 1, 3, 4).contiguous()
        v_p = batch['positive'].to(device).permute(2, 0, 1, 3, 4).contiguous()
        v_n = batch['negative'].to(device).permute(2, 0, 1, 3, 4).contiguous()

        # Anchor
        functional.reset_net(model)
        out_a = model(v_a)
        z_a = out_a.mean(dim=0) if out_a.dim() > 2 else out_a
        if z_a.dim() == 1:
            z_a = z_a.unsqueeze(0)
        z_a = F.normalize(z_a, p=2, dim=-1)

        # Positive
        functional.reset_net(model)
        out_p = model(v_p)
        z_p = out_p.mean(dim=0) if out_p.dim() > 2 else out_p
        if z_p.dim() == 1:
            z_p = z_p.unsqueeze(0)
        z_p = F.normalize(z_p, p=2, dim=-1)

        # Negative
        functional.reset_net(model)
        out_n = model(v_n)
        z_n = out_n.mean(dim=0) if out_n.dim() > 2 else out_n
        if z_n.dim() == 1:
            z_n = z_n.unsqueeze(0)
        z_n = F.normalize(z_n, p=2, dim=-1)

        dist_pos = F.pairwise_distance(z_a, z_p)
        dist_neg = F.pairwise_distance(z_a, z_n)

        rate_pos = torch.exp(-dist_pos) * 90.0 + 10.0
        rate_neg = torch.exp(-dist_neg) * 90.0 + 10.0

        current_batch_size = v_a.shape[1]
        for i in range(current_batch_size):
            t_id = batch['triplet_id'][i].item() if torch.is_tensor(
                batch['triplet_id']) else batch['triplet_id'][i]

            results.append({
                'triplet_id': int(t_id),
                'anchor': batch['video_names'][0][i] if isinstance(batch['video_names'][0], (list, tuple)) else batch['video_names'][i][0],
                'positive': batch['video_names'][1][i] if isinstance(batch['video_names'][1], (list, tuple)) else batch['video_names'][i][1],
                'negative': batch['video_names'][2][i] if isinstance(batch['video_names'][2], (list, tuple)) else batch['video_names'][i][2],
                'dist_pos': float(dist_pos[i].cpu().item()),
                'dist_neg': float(dist_neg[i].cpu().item()),
                'rate_pos_hz': float(rate_pos[i].cpu().item()),
                'rate_neg_hz': float(rate_neg[i].cpu().item())
            })

        del v_a, v_p, v_n, out_a, out_p, out_n, z_a, z_p, z_n, dist_pos, dist_neg, rate_pos, rate_neg
        torch.cuda.empty_cache()

    with open(save_json_path, 'w', encoding='utf-8') as f:
        json.dump(results, f, indent=4)

    print(f"💾 Saved {len(results)} items to: {save_json_path}")
    return results


train_dataset = CustomTripletDataset(
    json_path="/kaggle/input/datasets/mohammadmehranfar/triplet-dataset/Triplet_data/Triplet_train.json",
    video_dir="/kaggle/input/datasets/mohammadmehranfar/isik-dataset/dyad_videos_3000ms",
    num_frames=16
)
train_loader = DataLoader(train_dataset, batch_size=4,
                          num_workers=2, pin_memory=True, shuffle=True)

test_dataset = CustomTripletDataset(
    json_path="/kaggle/input/datasets/mohammadmehranfar/triplet-dataset/Triplet_data/Triplet_test.json",
    video_dir="/kaggle/input/datasets/mohammadmehranfar/isik-dataset/dyad_videos_3000ms",
    num_frames=16
)
test_loader = DataLoader(test_dataset, batch_size=4,
                         num_workers=2, pin_memory=True, shuffle=False)


save_path = '/kaggle/input/models/mohammadmehranfar/lorafbsnet-weights/pytorch/default/1/best_model.pth'
full_model = load_model_from_state_dict(save_path, num_classes=3)
extractor_model = LoRaFBSNetFeatureExtractor(full_model)
set_step_mode(extractor_model, 'm', (ConvRecurrentContainer,))

# optimizer = torch.optim.AdamW(
#     extractor_model.parameters(), lr=1e-4, weight_decay=1e-2)
# criterion = nn.TripletMarginLoss(margin=0.2, p=2)

# trained_extractor = train_lora_feature_extractor(
#     model=extractor_model,
#     train_loader=train_loader,
#     optimizer=optimizer,
#     criterion=criterion,
#     epochs=2,
# )

# checkpoint_path = "lora_trained_weights.pth"
# torch.save(trained_extractor.state_dict(), checkpoint_path)
# print(f"\n🎉 Saved Trained LoRA Weights to: {checkpoint_path}")

trained_weights_path = '/kaggle/input/models/mohammadmehranfar/lora-trained-weights/pytorch/default/1/lora_trained_weights.pth'
extractor_model.load_state_dict(torch.load(
    trained_weights_path, map_location=device))
extractor_model.to(device)
extractor_model.eval()
train_rates = extract_and_save_wang_inputs(
    model=extractor_model,
    dataloader=train_loader,
    save_json_path="wang_inputs_train.json",
)

test_rates = extract_and_save_wang_inputs(
    model=extractor_model,
    dataloader=test_loader,
    save_json_path="wang_inputs_test.json",
)

In [ ]:
print('Hello')